In [2]:
from dotenv import load_dotenv
import os

load_dotenv(".env")

api_key = os.getenv("OPENAI_API_KEY")

print("API key loaded successfully." if api_key else "API key not found.")

API key loaded successfully.


In [3]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

response = client.responses.create(
    model="gpt-5.6-luna",
    input="Say exactly: Olist AI connection successful."
)

print(response.output_text)

Olist AI connection successful.


In [4]:
import pandas as pd

reviews = pd.read_csv("../data/olist_order_reviews_dataset.csv")

review_ai = reviews[
    ["review_id", "order_id", "review_score", "review_comment_message"]
].copy()

review_ai = review_ai.dropna(
    subset=["review_comment_message"]
).copy()

review_ai["review_text"] = (
    review_ai["review_comment_message"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

review_ai = review_ai[
    review_ai["review_text"] != ""
].copy()

print("Reviews available for AI analysis:", len(review_ai))
print(review_ai[["review_score", "review_text"]].head(10))

Reviews available for AI analysis: 40950
    review_score                                        review_text
3              5              Recebi bem antes do prazo estipulado.
4              5  Parabéns lojas lannister adorei comprar pela I...
9              4  aparelho eficiente. no site a marca do aparelh...
12             4        Mas um pouco ,travando...pelo valor ta Boa.
15             5  Vendedor confiável, produto ok e entrega antes...
16             2  GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...
19             1                                            Péssimo
22             5                                       Loja nota 10
24             5              obrigado pela atençao amim dispensada
27             5  A compra foi realizada facilmente. A entrega f...


In [5]:
import json

sample_reviews = review_ai.head(10)

for _, row in sample_reviews.iterrows():

    prompt = f"""
You are analyzing customer reviews for an e-commerce company.

Read the review below. The review may be written in Portuguese.
Understand its meaning and return the classification in English.

Review:
{row["review_text"]}

Return ONLY valid JSON with these fields:

{{
    "sentiment": "Positive / Neutral / Negative",
    "issue_category": "Delivery / Product / Seller / Payment / Other",
    "severity": "Low / Medium / High",
    "main_complaint": "short description",
    "root_cause": "Logistics / Product Quality / Seller / Payment / Other / None",
    "theme": "short theme"
}}
"""

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    result = json.loads(response.output_text)

    print("\nReview:", row["review_text"])
    print(result)


Review: Recebi bem antes do prazo estipulado.
{'sentiment': 'Positive', 'issue_category': 'Delivery', 'severity': 'Low', 'main_complaint': 'No complaint; the order arrived well before the estimated deadline', 'root_cause': 'None', 'theme': 'Early delivery'}

Review: Parabéns lojas lannister adorei comprar pela Internet seguro e prático Parabéns a todos feliz Páscoa
{'sentiment': 'Positive', 'issue_category': 'Other', 'severity': 'Low', 'main_complaint': 'No complaint; the customer praises the safe and convenient online shopping experience.', 'root_cause': 'None', 'theme': 'Positive online shopping experience'}

Review: aparelho eficiente. no site a marca do aparelho esta impresso como 3desinfector e ao chegar esta com outro nome...atualizar com a marca correta uma vez que é o mesmo aparelho
{'sentiment': 'Positive', 'issue_category': 'Product', 'severity': 'Low', 'main_complaint': 'The brand name shown on the website differs from the name on the received device.', 'root_cause': 'Selle

In [7]:
negative_reviews = review_ai[
    review_ai["review_score"] <= 3
].copy()

print("Negative reviews:", len(negative_reviews))
print(negative_reviews[["review_score", "review_text"]].head(20))

Negative reviews: 14445
     review_score                                        review_text
16              2  GOSTARIA DE SABER O QUE HOUVE, SEMPRE RECEBI E...
19              1                                            Péssimo
29              1                Não gostei ! Comprei gato por lebre
32              1  Sempre compro pela Internet e a entrega ocorre...
39              1                       Nada de chegar o meu pedido.
51              1  recebi somente 1 controle Midea Split ESTILO. ...
68              1  O produto não chegou no prazo estipulado e cau...
73              3  Eu comprei duas unidades e só recebi uma e ago...
75              3  Produto bom, porém o que veio para mim não con...
76              1               Produto muito inferior, mal acabado.
87              3                                   Entrega no prazo
89              1          Pedi reembolso e sem resposta até momento
110             3  Produto chegou, mas meu PC não conseguiu recon...
115       

In [9]:
negative_sample = (
    negative_reviews
    .groupby("review_score", group_keys=False)
    .sample(n=17, random_state=42)
    .reset_index(drop=True)
)

print("Sample selected:", len(negative_sample))
print(negative_sample[["review_score", "review_text"]])

Sample selected: 51
    review_score                                        review_text
0              1                                  Demora de entrega
1              1  Depois que fiz a compra, passou 30 dias para a...
2              1  ainda não entregaram meu produto, apenas o out...
3              1  Não estou satisfeita, pois ainda não recebi o ...
4              1  Até hoje não recebi meu produto. Vou fazer uma...
5              1  O produto já passou da data de entrega e não c...
6              1  Até o momento não recebi o produto , não entra...
7              1  Produto veio faltando duas peças e eu pedi a t...
8              1  comprei dois produtos há mais de um mês e não ...
9              1  Solicitei informações acerca da entrega de meu...
10             1  Comprei dois produtos em um só número de pedid...
11             1  Recebi o produto dia 15.12.17 e encaminhei um ...
12             1  Comprei um produto,me mandaram um outro produt...
13             1  Falta a en

In [11]:
negative_sample.to_csv(
    "../data/negative_review_sample.csv",
    index=False
)

print("Negative review sample saved successfully.")

Negative review sample saved successfully.


In [12]:
# Create batches of 10 reviews each

batches = []

for start in range(0, len(negative_sample), 10):
    batch = negative_sample.iloc[start:start + 10]
    batches.append(batch)

print("Number of batches:", len(batches))
print("Reviews in first batch:", len(batches[0]))

Number of batches: 6
Reviews in first batch: 10


In [13]:
import json

ai_results = []

for batch_number, batch in enumerate(batches, start=1):

    reviews_text = ""

    for i, (_, row) in enumerate(batch.iterrows(), start=1):
        reviews_text += f"""
REVIEW {i}
Review ID: {row["review_id"]}
Order ID: {row["order_id"]}
Review Score: {row["review_score"]}
Review: {row["review_text"]}
"""

    prompt = f"""
You are an e-commerce customer experience analyst.

Analyze ALL customer reviews below. Reviews may be written in Portuguese.
Understand the meaning and classify each review.

{reviews_text}

For EACH review, return one JSON object.

Use ONLY these labels:

sentiment: Positive, Neutral, Negative
issue_category: Delivery, Product, Seller, Payment, Other
severity: Low, Medium, High
root_cause: Logistics, Product Quality, Seller, Payment, Other, None

Return ONLY a JSON array with this structure:

[
  {{
    "review_id": "...",
    "sentiment": "...",
    "issue_category": "...",
    "severity": "...",
    "main_complaint": "...",
    "root_cause": "...",
    "theme": "..."
  }}
]
"""

    response = client.responses.create(
        model="gpt-5.6-luna",
        input=prompt
    )

    batch_results = json.loads(response.output_text)
    ai_results.extend(batch_results)

    print(f"Batch {batch_number}/6 completed")

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-5.6-luna in organization org-RtMcKcHM0q8DVgU3Gle9mlmr on requests per day (RPD): Limit 50, Used 50, Requested 1. Please try again in 28m48s. Visit https://platform.openai.com/account/rate-limits to learn more. You can increase your rate limit by adding a payment method to your account at https://platform.openai.com/account/billing.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}